# Spaghettification near a black hole

*astro · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/astro/tidal_tensor)

A ring of particles falls toward a stellar-mass black hole. The difference in gravity across the ring is the tidal tensor $T_{ij} = -\partial_i\partial_j\Phi$, computed from its formula and as an autograd Hessian. It stretches the ring along the radial direction ($+2GM/r^3$) and squeezes it across ($-GM/r^3$ twice): spaghettification, which tears real bodies apart inside the tidal radius.

**Vector form**

$$\ddot{\boldsymbol\xi} = -(\boldsymbol\xi\cdot\nabla)\nabla\Phi,\qquad \Phi = -\frac{GM}{r},\qquad r_t = R\left(\frac{2M}{m}\right)^{1/3}$$

**Index (tensor) form**

$$\ddot\xi_{bi} = T_{ij}\,\xi_{bj},\qquad T_{ij} = -\partial_i\partial_j\Phi = \frac{GM}{r^5}\left(3x_ix_j - r^2\delta_{ij}\right)$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $x_i$ | `centre = y[0, 0]` | (3,) | position of the ring's centre (black hole at the origin) |
| $\xi_{bi}$ | `y[1 + n:, 0]` | (n, 3) | separation of ring particle $b$ from the centre |
| $x_ix_j$ | `einsum('i,j->ij', x, x)` | (3, 3) | outer product, no index summed |
| $T_{ij}$ | `tidal_tensor(x, GM)` | (3, 3) | tidal tensor from the formula |
| $-\partial_i\partial_j\Phi$ | `-vmap(jacrev(grad(potential)))(x)` | (P, 3, 3) | the same tensor as an autograd Hessian |
| $T_{ij}\xi_{bj}$ | `einsum('ij,bj->bi', T, xi)` | (n, 3) | repeated $j$ ⇒ summed: linear tidal acceleration |
| $T_{ii}$ | `torch.trace(T)` | () | $= -\nabla^2\Phi = 0$ in vacuum |
| $r_t$ | `tidal_radius(R, m/M)` | () | tidal (Roche) radius of a self-gravitating body |

Units: $G = c = 1$ and lengths in $GM/c^2$, so the horizon is at $r_s = 2$ (30 km for a 10-solar-mass black hole). The ring particles do not attract each other: the blue dots follow exact Newtonian gravity, the red curve the linear law $\ddot\xi_i = T_{ij}\xi_j$, and they agree until the ring is no longer small compared with $r$. The orange circle marks where a body of the ring's size and mass $m$ held together by its own gravity would be torn apart. For a stellar-mass black hole that radius lies far outside the horizon, so an infalling star (or astronaut) is shredded before it crosses; for a supermassive black hole it can lie inside the horizon. Gravity is Newtonian here; remarkably, the tidal tensor of a radially falling observer in general relativity has exactly the same eigenvalues $(2, -1, -1)\,GM/r^3$. The run stops before the horizon.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/astro/tidal_tensor/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

import torch

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    GM = tensors.GM
    y0 = tensors.initial_state(params["r0"], params["ring_radius"], params["n_particles"], dev)
    sol = solve.solve(y0, GM, params["r_stop"], params["n_steps"])

    # the tidal tensor at the start, from the formula and from autograd
    x0 = y0[:1, 0]  # (1, 3)
    T_formula = equation.tidal_tensor(x0, GM)[0]
    T_autograd = equation.tidal_tensor_autograd(x0, GM)[0]
    r0 = float(x0.norm())
    print(f"T_ij at r = {r0:g} (units GM/r³):\n{(T_formula * r0**3 / GM).cpu().numpy().round(6)}")
    print(f"autograd Hessian vs formula: max difference {(T_autograd - T_formula).abs().max():.1e}")
    print(f"trace T = {torch.trace(T_formula):.1e}   eigenvalues · r³/GM = {torch.linalg.eigvalsh(T_formula) * r0**3}")

    r_t = equation.tidal_radius(params["ring_radius"], params["mass_ratio"])
    along, across = solve.stretch(sol["xi_exact"])
    s = params["ring_radius"]
    print(f"tidal radius for a body with m/M = {params['mass_ratio']:g} and radius {s:g}: r_t = {r_t:.2f} GM/c²")
    print(f"at r = {params['r_stop']:g}: stretched ×{along[-1] / s:.2f} radially, squeezed ×{across[-1] / s:.2f}")
    return plot.animate(sol, GM, r_t, out_dir / "tidal_tensor.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/astro/tidal_tensor/equation.py
r"""Physics → tensor translation.

Near a black hole of mass M the pull of gravity changes quickly with distance. Two nearby particles at
x and x + ξ feel slightly different accelerations; to first order in the separation ξ (Taylor expansion)

    \ddot ξ_i = T_{ij}\, ξ_j,        T_{ij} = -\partial_i \partial_j Φ      (j summed)

T_{ij} is the tidal tensor. For a point mass, Φ = -GM/r, and

    T_{ij} = \frac{GM}{r^5}\,\left(3 x_i x_j - r^2 δ_{ij}\right)

Its trace vanishes outside the mass (∇²Φ = 0), and its eigenvalues are 2GM/r³ along the radial direction
(stretching) and −GM/r³ twice across it (squeezing): a falling body becomes a long thin noodle —
"spaghettification". Here T_{ij} is computed both from this formula and as the Hessian of Φ by automatic
differentiation (torch.func), which works for any potential.

A body held together by its own gravity (mass m, radius R) is torn apart when the tidal stretch 2GMR/r³
beats its surface gravity Gm/R², i.e. inside the tidal (Roche) radius

    r_t = R\,(2M/m)^{1/3}.
"""

import torch
from torch.func import grad, jacrev, vmap


def potential(x: torch.Tensor, GM: float) -> torch.Tensor:
    r"""Φ = -GM / \sqrt{x_j x_j}.   x: (..., 3) → (...)"""
    return -GM / torch.sqrt(torch.einsum("...j,...j->...", x, x))


def tidal_tensor(x: torch.Tensor, GM: float) -> torch.Tensor:
    r"""T_{ij} = GM (3 x_i x_j - r^2 δ_{ij}) / r^5.   x: (..., 3) → (..., 3, 3)"""
    r2 = torch.einsum("...j,...j->...", x, x)[..., None, None]  # (..., 1, 1)
    outer = torch.einsum("...i,...j->...ij", x, x)  # (..., 3, 3)
    delta = torch.eye(3, dtype=x.dtype, device=x.device)
    return GM * (3 * outer - r2 * delta) / r2**2.5


def tidal_tensor_autograd(x: torch.Tensor, GM: float) -> torch.Tensor:
    """T_ij = -∂_i∂_j Φ as the Hessian of the potential, by automatic differentiation.   (P, 3) → (P, 3, 3)"""
    return -vmap(jacrev(grad(lambda p: potential(p, GM))))(x)


def acceleration(x: torch.Tensor, GM: float) -> torch.Tensor:
    r"""Exact Newtonian pull \ddot x_{bi} = -GM (x_{bj} x_{bj})^{-3/2} x_{bi}.   (B, 3) → (B, 3)"""
    r2 = torch.einsum("bj,bj->b", x, x)
    return -GM * torch.einsum("b,bi->bi", r2**-1.5, x)


def tidal_acceleration(T: torch.Tensor, xi: torch.Tensor) -> torch.Tensor:
    r"""\ddot ξ_{bi} = T_{ij} ξ_{bj} (j summed): linear tidal law for each ring particle b.   (3, 3), (B, 3) → (B, 3)"""
    return torch.einsum("ij,bj->bi", T, xi)


def rhs(y: torch.Tensor, GM: float, n: int) -> torch.Tensor:
    """dy/dt for y (1 + 2n, 2, 3): [ring centre, n ring particles (exact), n separations ξ (tidal law)]."""
    x, v = y[:, 0], y[:, 1]
    a_exact = acceleration(x[: 1 + n], GM)  # centre and ring particles, full gravity
    a_tidal = tidal_acceleration(tidal_tensor(x[0], GM), x[1 + n :])  # separations from the centre, linearised
    return torch.stack([v, torch.cat([a_exact, a_tidal])], dim=1)


def tidal_radius(R_body: float, mass_ratio: float) -> float:
    """r_t = R (2M/m)^{1/3}, mass_ratio = m/M."""
    return R_body * (2 / mass_ratio) ** (1 / 3)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/astro/tidal_tensor/tensors.py
"""Build the concrete tensors: a ring of test particles dropped from rest toward the black hole.

Units: G = c = 1 and lengths in GM/c², so the black hole has GM = 1 and its horizon (Schwarzschild radius)
is at r_s = 2. For a 10-solar-mass black hole one length unit is 15 km and one time unit 49 µs.
"""

import math

import torch

GM = 1.0


def initial_state(r0: float, ring_radius: float, n: int, device: torch.device) -> torch.Tensor:
    """y0 of shape (1 + 2n, 2, 3), everything at rest.

    row 0          centre of the ring, at (r0, 0, 0)
    rows 1 … n     ring particles at the centre + ξ_b, followed exactly
    rows n+1 … 2n  the separations ξ_b themselves, evolved with the linear tidal law
    """
    phi = torch.arange(n, dtype=torch.float64) * 2 * math.pi / n
    xi = ring_radius * torch.stack([torch.cos(phi), torch.sin(phi), torch.zeros_like(phi)], dim=-1)  # (n, 3)
    centre = torch.tensor([[r0, 0.0, 0.0]], dtype=torch.float64)
    x = torch.cat([centre, centre + xi, xi])
    return torch.stack([x, torch.zeros_like(x)], dim=1).to(device)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/astro/tidal_tensor/solve.py
"""Drop the ring with RK4 until its centre reaches r_stop, and compare exact and tidal-tensor deformations."""

import math

import torch

from problems._common.integrators import integrate

from .equation import rhs


def fall_time(r0: float, r: float, GM: float) -> float:
    """Newtonian time to fall from rest at r0 down to r (radial Kepler orbit, 'cycloid' solution)."""
    eta = math.acos(2 * r / r0 - 1)  # development angle: r = r0 (1 + cos η) / 2
    return math.sqrt(r0**3 / (8 * GM)) * (eta + math.sin(eta))


def solve(y0: torch.Tensor, GM: float, r_stop: float, n_steps: int) -> dict:
    """Return times (T,), the ring in absolute coordinates (T, n, 3) and its deformation, exact and linear."""
    n = (y0.shape[0] - 1) // 2
    r0 = float(y0[0, 0].norm())
    t_end = fall_time(r0, r_stop, GM)
    dt = t_end / n_steps
    traj = integrate(lambda y, t: rhs(y, GM, n), y0, dt, n_steps)  # (T, 1 + 2n, 2, 3)
    x = traj[:, :, 0]  # positions only
    centre = x[:, 0]  # (T, 3)
    return {
        "t": torch.arange(n_steps + 1, dtype=y0.dtype, device=y0.device) * dt,
        "centre": centre,
        "ring": x[:, 1 : 1 + n],  # (T, n, 3)
        "xi_exact": x[:, 1 : 1 + n] - centre[:, None],  # (T, n, 3) separation from the centre, full gravity
        "xi_tidal": x[:, 1 + n :],  # (T, n, 3) separation from the linear tidal law
    }


def stretch(xi: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Half-length of the ring along x (radial) and along y (transverse), (T,) each."""
    return xi[..., 0].abs().amax(-1), xi[..., 1].abs().amax(-1)


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/astro/tidal_tensor/plot.py
"""Animate the falling ring: the whole fall, a zoom on its shape (exact vs tidal tensor), and its size vs r."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from problems._common.plotting import ACCENT, save_gif

from .equation import tidal_tensor


def frames_by_radius(r: np.ndarray, n_frames: int) -> list[int]:
    """Steps where r crosses evenly spaced values of log r: the fall speeds up, the animation should not."""
    from problems._common.plotting import frame_indices

    n_frames = len(frame_indices(len(r), n_frames))  # respects fast mode
    targets = np.geomspace(r[0], r[-1], n_frames)
    order = np.argsort(r)  # r decreases with time
    steps = np.interp(targets, r[order], order)
    return sorted({int(round(s)) for s in steps})


def animate(sol: dict, GM: float, r_t: float, path: Path) -> Path:
    centre = sol["centre"].cpu().numpy()  # (T, 3)
    ring = sol["ring"].cpu().numpy()  # (T, n, 3)
    xi_e, xi_l = sol["xi_exact"].cpu().numpy(), sol["xi_tidal"].cpu().numpy()
    r = np.linalg.norm(centre, axis=-1)
    s0 = np.linalg.norm(xi_e[0], axis=-1).max()
    along_e, across_e = np.abs(xi_e[..., 0]).max(-1), np.abs(xi_e[..., 1]).max(-1)
    along_l, across_l = np.abs(xi_l[..., 0]).max(-1), np.abs(xi_l[..., 1]).max(-1)
    T = tidal_tensor(sol["centre"], GM).cpu().numpy()  # (T, 3, 3)

    fig = plt.figure(figsize=(8, 5))
    gs = fig.add_gridspec(2, 2, height_ratios=[1, 1.5])
    ax = fig.add_subplot(gs[0, :])
    ax_z = fig.add_subplot(gs[1, 0])
    ax_s = fig.add_subplot(gs[1, 1])

    # --- the whole fall, true scale ---
    ax.set_xlim(-3, r[0] + 2)
    ax.set_ylim(-4, 4)
    ax.set_aspect("equal", adjustable="box")
    ax.add_patch(plt.Circle((0, 0), 2 * GM, color="black", zorder=4))
    ax.add_patch(plt.Circle((0, 0), r_t, fill=False, ls="--", color="#f59e0b", lw=1.2))
    ax.text(r_t, 1.2, "tidal radius r_t", color="#b45309", fontsize=8)
    ax.text(2.5, -3.2, "horizon r_s = 2GM/c²", fontsize=8)
    ax.set_xlabel("x  (GM/c²)")
    blob = ax.fill([], [], color=ACCENT[0], ec=ACCENT[0], lw=2.5, zorder=5)[0]
    path_line = ax.plot([], [], color="#9ca3af", lw=0.8)[0]

    # --- zoom on the deformation, in the ring's own (falling) frame ---
    phi = np.linspace(0, 2 * np.pi, 100)
    ax_z.plot(s0 * np.cos(phi), s0 * np.sin(phi), "--", color="#9ca3af", lw=1, label="start")
    exact = ax_z.plot([], [], "o", ms=3, color=ACCENT[0], label="exact gravity")[0]
    linear = ax_z.plot([], [], "-", color=ACCENT[1], lw=1.2, label="tidal tensor T·ξ")[0]
    ax_z.set_aspect("equal", adjustable="box")
    ax_z.set_xlabel("ξ_x  (radial, black hole ←)")
    ax_z.set_ylabel("ξ_y")
    ax_z.legend(fontsize=7, loc="upper right")
    title_z = ax_z.set_title("", fontsize=8)

    # --- half-lengths against distance ---
    ax_s.loglog(r, along_e / s0, color=ACCENT[0], lw=1, alpha=0.35)
    ax_s.loglog(r, across_e / s0, color=ACCENT[2], lw=1, alpha=0.35)
    ax_s.set_xlim(r[0] * 1.05, r[-1] * 0.95)  # r decreases to the right, as time goes on
    ax_s.axvline(r_t, color="#f59e0b", ls="--", lw=1)
    run_along = ax_s.loglog([], [], color=ACCENT[0], lw=2, label="radial (stretch)")[0]
    run_across = ax_s.loglog([], [], color=ACCENT[2], lw=2, label="transverse (squeeze)")[0]
    run_lin = ax_s.loglog([], [], ":", color=ACCENT[1], lw=1.5, label="tidal tensor")[0]
    ax_s.loglog([], [], ":", color=ACCENT[1], lw=1.5)
    ax_s.set_xlabel("distance r  (GM/c²)")
    ax_s.set_ylabel("half-length / initial radius")
    ax_s.legend(fontsize=7, loc="upper left")
    ticks = [v for v in (100, 50, 40, 20, 10, 5, 3, 2) if r[-1] <= v <= r[0]]
    ax_s.set_xticks(ticks, [f"{v:g}" for v in ticks])
    ax_s.minorticks_off()

    def update(step: int):
        outline = ring[step, :, :2]
        blob.set_xy(outline)
        path_line.set_data(centre[: step + 1, 0], centre[: step + 1, 1])
        exact.set_data(xi_e[step, :, 0], xi_e[step, :, 1])
        loop = np.vstack([xi_l[step], xi_l[step, :1]])
        linear.set_data(loop[:, 0], loop[:, 1])
        reach = 1.15 * max(s0, np.abs(xi_e[step, :, :2]).max())
        ax_z.set_xlim(-reach, reach)
        ax_z.set_ylim(-reach, reach)
        ev = np.linalg.eigvalsh(T[step]) * r[step] ** 3 / GM
        title_z.set_text(f"r = {r[step]:.1f}:  eig(T)·r³/GM = {ev[2]:+.2f}, {ev[1]:+.2f}, {ev[0]:+.2f}")
        run_along.set_data(r[: step + 1], along_e[: step + 1] / s0)
        run_across.set_data(r[: step + 1], across_e[: step + 1] / s0)
        stacked = np.concatenate([along_l[: step + 1], [np.nan], across_l[: step + 1]]) / s0
        run_lin.set_data(np.concatenate([r[: step + 1], [np.nan], r[: step + 1]]), stacked)

    fig.tight_layout()
    return save_gif(fig, update, frames_by_radius(r, 90), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'r0': 40.0,
    'r_stop': 3.0,
    'ring_radius': 0.2,
    'n_particles': 24,
    'mass_ratio': 0.0001,
    'n_steps': 3000
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.astro.tidal_tensor")]:
    del sys.modules[name]
main = importlib.import_module("problems.astro.tidal_tensor.main")
problem_dir = Path("problems/astro/tidal_tensor")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
